# Outline

This notebook use Claude Code for EDA and preprocess function. Because I am not working as data engineer this helps me a lot. Claude give me a lot of different ways to analyze data and I also learn a lot from this.

Notebook Outline

1. Load Data And Library
2. Preprocess Data
3. Model training
4. Submission

# Big update: Claude Code

I use claude code to analyze data with me and create a report.

link: [claude code EDA](https://github.com/wesleyhuan/Kaggle_S6E5_EDA)


Key summary:

* The dataset has 439,140 rows x 16 columns, using 127.3 MB of memory
* No missing values at all (typical of an already-cleaned Kaggle dataset)
* The target PitNextLap is binary with a class imbalance of roughly 80 : 20 (negative : positive)
* Driver is a high-cardinality categorical (887 values) mixing 3-letter real abbreviations with D000-style anonymized codes
* LapTime has an extreme tail (max 2507s) caused by safety-car or red-flag
* laps TyreLife, LapNumber, Stint and RaceProgress are positively correlated with the target (r ~ 0.19~0.27)
* Cumulative_Degradation is negatively correlated with the target (r ~ -0.17)
* Pit rate differs hugely by Compound: HARD 32.8%, MEDIUM 10.1%, WET 2.5%
* Race includes Pre-Season Testing, whose pitting logic differs from real races
* Pit probability peaks mid-race (RaceProgress 0.4~0.6), forming an inverted-U shape

# Load Data And Library

In [1]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold


import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

import optuna

In [2]:
# config
class CFG:
    train_csv = '/kaggle/input/competitions/playground-series-s6e5/train.csv'
    test_csv = '/kaggle/input/competitions/playground-series-s6e5/test.csv'
    sample_submission_csv = '/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv'
    N_FOLDS = 5
    RANDOM_SEED = 42
    
#torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Preprocess Function (Claude Code)

In [3]:
"""
F1 PitNextLap - Complete Data Preprocessing Class
=================================================

Consolidates all 12 preprocessing recommendations from the EDA report
into a single sklearn-style class.

Usage:

    from f1_preprocessor import F1Preprocessor
    import pandas as pd

    df_train = pd.read_csv('train.csv')
    df_test  = pd.read_csv('test.csv')

    pre = F1Preprocessor(target='PitNextLap', drop_pre_season=True)
    X_train, y_train = pre.fit_transform(df_train, return_y=True)
    X_test           = pre.transform(df_test)

    # Then feed straight into a model
    import lightgbm as lgb
    model = lgb.LGBMClassifier(scale_pos_weight=pre.scale_pos_weight_)
    model.fit(X_train, y_train)

Design principles:
  * fit() must only see the train set. Every statistic (median lap time,
    target encoding, frequency tables) is computed during fit and stored,
    so transform() only does table lookups -> prevents data leakage.
  * Tree-model friendly: keeps original categorical columns and adds
    ordinal, frequency, and target-encoding features.
  * For linear models, apply a StandardScaler separately; this class
    does not scale features.
"""
class F1Preprocessor(BaseEstimator, TransformerMixin):
    # ---- Domain constants -------------------------------------------------
    COMPOUND_HARDNESS = {       # dry-tyre hardness ranking (wet tyres = 0)
        'SOFT': 1, 'MEDIUM': 2, 'HARD': 3, 'INTERMEDIATE': 0, 'WET': 0,
    }
    EXPECTED_STINT = {          # empirical "expected usable laps" per compound
        'SOFT': 18, 'MEDIUM': 25, 'HARD': 35,
        'INTERMEDIATE': 20, 'WET': 15,
    }
    WET_COMPOUNDS = {'INTERMEDIATE', 'WET'}
    DRIVER_RARE_THRESHOLD = 50  # drivers with < 50 samples are grouped as OTHER
    LAPTIME_CLIP_QUANTILES = (0.005, 0.995)

    # ---- Interface --------------------------------------------------------
    def __init__(self,
                 target: str = 'PitNextLap',
                 drop_pre_season: bool = True,
                 use_target_encoding: bool = True,
                 te_smoothing: float = 20.0,
                 te_n_splits: int = 5,
                 add_lag_features: bool = True,
                 random_state: int = CFG.RANDOM_SEED):
        self.target = target
        self.drop_pre_season = drop_pre_season
        self.use_target_encoding = use_target_encoding
        self.te_smoothing = te_smoothing
        self.te_n_splits = te_n_splits
        self.add_lag_features = add_lag_features
        self.random_state = random_state

    # =====================================================================
    # fit
    # =====================================================================
    def fit(self, df: pd.DataFrame, y=None):
        """Learn parameters from train. y is taken from df[target] automatically."""
        if self.drop_pre_season:
            df = df[df['Race'] != 'Pre-Season Testing'].copy()
        else:
            df = df.copy()

        y = df[self.target].astype(int).values
        self.global_pit_rate_ = float(np.mean(y))
        n_neg = int((y == 0).sum())
        n_pos = int((y == 1).sum())
        self.scale_pos_weight_ = n_neg / max(n_pos, 1)

        # 1. Median lap-time baseline per race (used for relative lap time)
        self.lap_baseline_ = (
            df.groupby(['Race', 'Year'])['LapTime (s)'].median().to_dict()
        )

        # 2. Lap-time clip bounds (also per Race x Year)
        lo_q, hi_q = self.LAPTIME_CLIP_QUANTILES
        self.lap_clip_ = (
            df.groupby(['Race', 'Year'])['LapTime (s)']
              .agg(lo=lambda s: s.quantile(lo_q),
                   hi=lambda s: s.quantile(hi_q))
              .to_dict('index')
        )

        # 3. Driver frequency table + whitelist of non-rare drivers
        drv_counts = df['Driver'].value_counts()
        self.driver_freq_ = drv_counts.to_dict()
        self.driver_keep_ = set(drv_counts[drv_counts >= self.DRIVER_RARE_THRESHOLD].index)

        # 4. Target encoding (smoothed; statistics computed on train only)
        if self.use_target_encoding:
            self.te_driver_ = self._fit_target_encoding(df, 'Driver', y)
            self.te_race_   = self._fit_target_encoding(df, 'Race', y)
        else:
            self.te_driver_ = self.te_race_ = {}

        # 5. Max stint length per race x compound (for derived features)
        self.compound_race_max_stint_ = (
            df.groupby(['Compound', 'Race'])['TyreLife'].max().to_dict()
        )

        self.feature_names_ = None  # filled in during transform
        return self

    def _fit_target_encoding(self, df, col, y):
        """Smoothed target encoding (shrinks rare categories toward the prior)."""
        prior = self.global_pit_rate_
        sm = self.te_smoothing
        agg = df.groupby(col)[self.target].agg(['mean', 'count'])
        smoothed = (agg['mean'] * agg['count'] + prior * sm) / (agg['count'] + sm)
        return smoothed.to_dict()

    # =====================================================================
    # transform
    # =====================================================================
    def transform(self, df: pd.DataFrame, return_y: bool = False):
        df = df.copy()
        # Pre-Season Testing: do not drop in transform (test set may contain it),
        # just add a flag column
        df['is_pre_season'] = (df['Race'] == 'Pre-Season Testing').astype(int)

        # ---------- Numeric cleaning ----------
        df = self._clip_extreme_laptime(df)
        df = self._add_relative_laptime(df)

        # ---------- Categorical encoding ----------
        df = self._encode_compound(df)
        df = self._encode_driver(df)
        if self.use_target_encoding:
            df['race_te']   = df['Race'].map(self.te_race_).fillna(self.global_pit_rate_)
            df['driver_te'] = df['Driver'].map(self.te_driver_).fillna(self.global_pit_rate_)

        # ---------- Domain-derived features ----------
        df = self._add_tyre_features(df)
        df = self._add_position_features(df)
        df = self._add_progress_features(df)

        # ---------- Time-series features ----------
        if self.add_lag_features:
            df = self._add_lag_features(df)

        # ---------- Split into X / y ----------
        drop_cols = ['id', 'Driver', 'Compound', 'Race']
        if return_y:
            y = df[self.target].astype(int).values
            X = df.drop(columns=drop_cols + [self.target], errors='ignore')
            self.feature_names_ = list(X.columns)
            return X, y
        X = df.drop(columns=drop_cols + [self.target], errors='ignore')
        self.feature_names_ = list(X.columns)
        return X

    def fit_transform(self, df, y=None, return_y: bool = True):
        self.fit(df)
        # fit() already drops pre-season; transform must use the same rows
        if self.drop_pre_season:
            df = df[df['Race'] != 'Pre-Season Testing'].copy()
        return self.transform(df, return_y=return_y)

    # =====================================================================
    # Detailed steps
    # =====================================================================
    def _clip_extreme_laptime(self, df):
        """Clip each race's lap times to the 0.5%~99.5% range and flag anomalies."""
        global_lo = np.quantile(list(self.lap_baseline_.values()), 0.01)
        global_hi = np.quantile(list(self.lap_baseline_.values()), 0.99) * 2

        def _clip(row):
            key = (row['Race'], row['Year'])
            bounds = self.lap_clip_.get(key)
            if bounds is None:
                return np.clip(row['LapTime (s)'], global_lo, global_hi)
            return np.clip(row['LapTime (s)'], bounds['lo'], bounds['hi'])

        # For speed, look up the bounds via map then compute in one pass
        lo = df.set_index(['Race', 'Year']).index.map(
            lambda k: self.lap_clip_.get(k, {}).get('lo', global_lo))
        hi = df.set_index(['Race', 'Year']).index.map(
            lambda k: self.lap_clip_.get(k, {}).get('hi', global_hi))
        original = df['LapTime (s)'].values
        clipped = np.clip(original, lo.astype(float), hi.astype(float))
        df['is_anomaly_lap'] = (original != clipped).astype(int)
        df['LapTime_clipped'] = clipped
        df['LapTime_log'] = np.log1p(np.clip(clipped, 1, None))
        return df

    def _add_relative_laptime(self, df):
        """Lap time relative to that race's median lap time."""
        baseline = df.set_index(['Race', 'Year']).index.map(
            lambda k: self.lap_baseline_.get(k, np.nan))
        baseline = pd.Series(baseline, index=df.index, dtype=float)
        # If the test set has a race unseen in train, fall back to the global median
        baseline = baseline.fillna(np.median(list(self.lap_baseline_.values())))
        df['lap_baseline']  = baseline.values
        df['LapTime_diff']  = df['LapTime_clipped'] - baseline
        df['LapTime_ratio'] = df['LapTime_clipped'] / baseline
        return df

    def _encode_compound(self, df):
        df['compound_hardness'] = df['Compound'].map(self.COMPOUND_HARDNESS).fillna(0).astype(int)
        df['is_wet_tyre'] = df['Compound'].isin(self.WET_COMPOUNDS).astype(int)
        # Keep a one-hot copy as well, useful for linear models
        for c in ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']:
            df[f'compound_{c}'] = (df['Compound'] == c).astype(int)
        return df

    def _encode_driver(self, df):
        df['driver_freq'] = df['Driver'].map(self.driver_freq_).fillna(0).astype(int)
        df['driver_is_real'] = df['Driver'].astype(str).str.match(r'^[A-Z]{3}$').astype(int)
        df['driver_clean'] = df['Driver'].where(df['Driver'].isin(self.driver_keep_), 'OTHER')
        # driver_clean can be fed to LightGBM as a categorical directly;
        # here we label-encode it
        df['driver_clean_id'] = df['driver_clean'].astype('category').cat.codes
        return df.drop(columns=['driver_clean'])

    def _add_tyre_features(self, df):
        # Median life per compound
        med_life = {'SOFT': 10, 'MEDIUM': 11, 'HARD': 17, 'INTERMEDIATE': 12, 'WET': 9}
        df['tyre_life_ratio'] = df['TyreLife'] / df['Compound'].map(med_life).fillna(15)
        df['tyre_life_vs_expected'] = df['TyreLife'] / df['Compound'].map(self.EXPECTED_STINT).fillna(25)
        # Historical longest stint for the same race-compound pair
        key = list(zip(df['Compound'], df['Race']))
        df['compound_race_max_stint'] = [self.compound_race_max_stint_.get(k, np.nan) for k in key]
        df['compound_race_max_stint'] = df['compound_race_max_stint'].fillna(df['TyreLife'].median())
        df['tyre_life_vs_max'] = df['TyreLife'] / df['compound_race_max_stint'].clip(lower=1)
        return df

    def _add_position_features(self, df):
        df['in_points']        = (df['Position'] <= 10).astype(int)
        df['top3']             = (df['Position'] <= 3).astype(int)
        df['gained_pos']       = (df['Position_Change'] > 0).astype(int)
        df['lost_pos']         = (df['Position_Change'] < 0).astype(int)
        df['pos_gap_to_leader']= df['Position'] - 1
        return df

    def _add_progress_features(self, df):
        df['lap_remaining_pct'] = (1 - df['RaceProgress']).clip(lower=0)
        # Mid-race window (the range where pit stops happen most often)
        df['is_mid_race'] = ((df['RaceProgress'] >= 0.30) &
                             (df['RaceProgress'] <= 0.70)).astype(int)
        df['stint_lap_norm'] = df['TyreLife'] / df['Stint'].clip(lower=1)
        return df

    def _add_lag_features(self, df):
        """Build lag / rolling features grouped by (Race, Year, Driver, Stint).
        Note: df must contain LapNumber, and row order must be preserved."""
        df = df.sort_values(['Race', 'Year', 'Driver', 'Stint', 'LapNumber'])
        g = df.groupby(['Race', 'Year', 'Driver', 'Stint'], sort=False)

        for k in (1, 2, 3):
            df[f'lap_lag{k}']      = g['LapTime_clipped'].shift(k)
            df[f'lap_diff_lag{k}'] = df['LapTime_clipped'] - df[f'lap_lag{k}']

        df['lap_roll3_mean'] = g['LapTime_clipped'].transform(
            lambda s: s.rolling(3, min_periods=1).mean())
        df['lap_roll3_std']  = g['LapTime_clipped'].transform(
            lambda s: s.rolling(3, min_periods=2).std())
        df['lap_roll5_mean'] = g['LapTime_clipped'].transform(
            lambda s: s.rolling(5, min_periods=1).mean())

        # NaN (first few laps of each stint) -> 0; also add a mask column
        # so the model knows the value was missing
        for c in ['lap_lag1','lap_lag2','lap_lag3',
                  'lap_diff_lag1','lap_diff_lag2','lap_diff_lag3',
                  'lap_roll3_std']:
            df[f'{c}_isna'] = df[c].isna().astype(int)
            df[c] = df[c].fillna(0)
        df['lap_roll3_mean'] = df['lap_roll3_mean'].fillna(df['LapTime_clipped'])
        df['lap_roll5_mean'] = df['lap_roll5_mean'].fillna(df['LapTime_clipped'])

        return df.sort_index()

    # =====================================================================
    # CV split helper: avoid time-series leakage
    # =====================================================================
    @staticmethod
    def make_groups(df: pd.DataFrame) -> pd.Series:
        """Return a group id made of Race+Year, for use with GroupKFold."""
        return (df['Race'].astype(str) + '_' + df['Year'].astype(str))

# Preprocess data

In [4]:

# ===========================================================================
# Preprocess data (Claude code)
# ===========================================================================

df_train = pd.read_csv(CFG.train_csv)
pre = F1Preprocessor(
    target='PitNextLap',
    drop_pre_season=True,
    use_target_encoding=True,
    add_lag_features=True
)

X, y = pre.fit_transform(df_train, return_y=True)
print(f'Train shape: {X.shape}, Pos rate: {y.mean():.4f}, SPW: {pre.scale_pos_weight_:.2f}')

Train shape: (416648, 58), Pos rate: 0.2018, SPW: 3.96


# Model Training

In [5]:

# Retrieve groups for GroupKFold
groups = pre.make_groups(df_train[df_train['Race'] != 'Pre-Season Testing'])
gkf = GroupKFold(n_splits=CFG.N_FOLDS)

models = []  # List to store our trained models
scores = []

# Suppress lightgbm warnings during Optuna search
optuna.logging.set_verbosity(optuna.logging.INFO)

def objective(trial):
    # 1. Define the hyperparameter search space
    params = {
        # Fix n_estimators high, rely on early_stopping to find the exact tree count
        "n_estimators": 1500, 
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        
        # Tree Structure
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        # num_leaves must be tuned carefully. A safe upper bound is 2^max_depth
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 500),
        
        # Regularization & Sampling
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        
        # Fixed parameters
        "scale_pos_weight": pre.scale_pos_weight_, 
        "random_state": CFG.RANDOM_SEED,
        "n_jobs": -1,
        "verbose": -1
    }
    
    # Structural constraint: num_leaves cannot exceed 2^max_depth
    if params["num_leaves"] > (2 ** params["max_depth"]):
        params["num_leaves"] = (2 ** params["max_depth"]) - 1
        
    gkf = GroupKFold(n_splits=CFG.N_FOLDS)
    cv_scores = []
    
    # 2. Run your exact CV Loop
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups)):
        model = lgb.LGBMClassifier(**params)
        
        model.fit(
            X.iloc[tr], y[tr],
            eval_set=[(X.iloc[va], y[va])],
            # Use verbose=False so your console isn't flooded during 50 trials
            callbacks=[lgb.early_stopping(50, verbose=False)] 
        )
        
        p = model.predict_proba(X.iloc[va])[:, 1]
        auc = roc_auc_score(y[va], p)
        cv_scores.append(auc)
        
    # 3. Return the mean AUC
    return np.mean(cv_scores)

print("--- Starting Optuna Hyperparameter Optimization ---")
# 4. Create and run the study
study = optuna.create_study(direction="maximize", study_name="LGBM_PitStop_Tuning")

# 50 trials is usually the sweet spot. 
# Anything over 100 risks overfitting to your validation set.
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n=== Best Parameters Found ===")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"Best Mean AUC: {study.best_value:.4f}")

[I 2026-05-25 15:17:54,551] A new study created in memory with name: LGBM_PitStop_Tuning


--- Starting Optuna Hyperparameter Optimization ---


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-25 15:19:59,486] Trial 0 finished with value: 0.9192552268892447 and parameters: {'learning_rate': 0.07181889539133736, 'max_depth': 8, 'num_leaves': 50, 'min_child_samples': 74, 'subsample': 0.8591060918290943, 'colsample_bytree': 0.9115778287985568, 'reg_alpha': 1.4667497911939151e-08, 'reg_lambda': 7.865456784594324e-06}. Best is trial 0 with value: 0.9192552268892447.
[I 2026-05-25 15:21:53,618] Trial 1 finished with value: 0.9184800821392216 and parameters: {'learning_rate': 0.01606541820375628, 'max_depth': 7, 'num_leaves': 69, 'min_child_samples': 391, 'subsample': 0.8055157655539563, 'colsample_bytree': 0.9731749524428395, 'reg_alpha': 3.413829020878539e-08, 'reg_lambda': 0.014844805693564705}. Best is trial 0 with value: 0.9192552268892447.
[I 2026-05-25 15:23:19,254] Trial 2 finished with value: 0.9151736356797283 and parameters: {'learning_rate': 0.04306713508892215, 'max_depth': 4, 'num_leaves': 99, 'min_child_samples': 439, 'subsample': 0.8784229249332254, 'cols

# Submission

# OPTUNA

In [6]:
# 1. Re-initialize the best parameters found by Optuna
best_params = study.best_params
best_params['n_estimators'] = 1500  # Set a high number for final training
best_params['random_state'] = CFG.RANDOM_SEED

# 2. Preprocess the test data
df_test = pd.read_csv(CFG.test_csv)
X_test = pre.transform(df_test)  # Important: use transform(), NOT fit_transform()

# 3. Train final model(s)
# Recommended: Train on full training data
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# 4. Predict
test_preds = final_model.predict_proba(X_test)[:, 1]

# 5. Create Submission
sample_sub = pd.read_csv(CFG.sample_submission_csv)
sample_sub['PitNextLap'] = test_preds
sample_sub.to_csv('submission.csv', index=False)

print("Submission file saved successfully.")

Submission file saved successfully.


In [7]:
sample_sub.head()

,id,PitNextLap
0,439140,0.003123
1,439141,0.002808
2,439142,0.001831
3,439143,0.177588
4,439144,0.765828
